# 07 Execution manifest (dry-run only)

Build rollback-ready manifests from the latest dry-run planner output.

This notebook still makes **zero file changes**. It only separates rows into:
- executable manifest
- keep register
- review queue
- blocked rows
- rollback manifest


In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'
PLAN_PATH = None  # set explicitly if you want; otherwise latest plan_dry_run_*.parquet is used

print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUTS_DIR  =', OUTPUTS_DIR)


PROJECT_ROOT = c:\00_Developement\sch-file-organizer
OUTPUTS_DIR  = c:\00_Developement\sch-file-organizer\data\outputs


In [2]:
from src.reporting import find_latest_output
from src.executor import ManifestConfig, build_execution_bundle, save_manifest_bundle, manifest_summary

if PLAN_PATH is None:
    PLAN_PATH = find_latest_output(OUTPUTS_DIR, 'plan_dry_run_')

print('PLAN_PATH =', PLAN_PATH)
plan = pd.read_parquet(PLAN_PATH)
print('Rows:', len(plan))
preview_cols = [c for c in ['relative_path', 'planner_action', 'planner_ready', 'planner_needs_user_input', 'planner_target_relative_path'] if c in plan.columns]
display(plan[preview_cols].head(15))


PLAN_PATH = c:\00_Developement\sch-file-organizer\data\outputs\plan_dry_run_20260307_094350.parquet
Rows: 4


,relative_path,planner_action,planner_ready,planner_needs_user_input,planner_target_relative_path
0,docs/Thumbs.db,archive_or_delete_review,False,True,None
1,docs/a.txt,review_special_folder_policy,False,True,None
2,docs/PV15p473-01_PM_PER_environmental-approval...,manual_review,False,True,None
3,docs/b.txt,manual_review,False,True,None


## Manifest settings

Leave these conservative at first. In particular, keep collision blocking enabled.


In [3]:
EXECUTABLE_ACTIONS = (
    'move_to_policy_folder',
    'move_to_superseded_folder',
    'move_to_duplicate_folder',
    'move_to_deprecated_folder',
)
INCLUDE_KEEP_REGISTER = True
BLOCK_TARGET_COLLISIONS = True
REQUIRE_TARGET_PATH = True
REQUIRE_CHANGED_PATH = True

config = ManifestConfig(
    executable_actions=EXECUTABLE_ACTIONS,
    include_keep_register=INCLUDE_KEEP_REGISTER,
    block_target_collisions=BLOCK_TARGET_COLLISIONS,
    require_target_path=REQUIRE_TARGET_PATH,
    require_changed_path=REQUIRE_CHANGED_PATH,
)
config


ManifestConfig(executable_actions=('move_to_policy_folder', 'move_to_superseded_folder', 'move_to_duplicate_folder', 'move_to_deprecated_folder'), include_keep_register=True, block_target_collisions=True, require_target_path=True, require_changed_path=True)

In [4]:
bundle = build_execution_bundle(plan, config=config)
summary = manifest_summary(bundle)
summary


{'executable_rows': 0,
 'keep_rows': 0,
 'review_rows': 4,
 'blocked_rows': 0,
 'rollback_rows': 0}

In [5]:
display(bundle.executable_manifest[['relative_path', 'planner_action', 'execution_target_relative_path']].head(20))
display(bundle.keep_register[['relative_path', 'planner_action', 'keep_status']].head(20))
display(bundle.blocked_manifest[['relative_path', 'planner_action', 'execution_block_reason']].head(20))
display(bundle.review_queue[['relative_path', 'planner_action', 'planner_reason']].head(20))


KeyError: "['execution_target_relative_path'] not in index"

In [6]:
STAMP = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
saved = save_manifest_bundle(bundle, OUTPUTS_DIR, stem=STAMP)
saved


{'executable_manifest': (WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_ready_20260307_095943.csv'),
  WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_ready_20260307_095943.parquet')),
 'keep_register': (WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_keep_20260307_095943.csv'),
  WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_keep_20260307_095943.parquet')),
 'review_queue': (WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_review_20260307_095943.csv'),
  WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_review_20260307_095943.parquet')),
 'blocked_manifest': (WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_blocked_20260307_095943.csv'),
  WindowsPath('c:/00_Developement/sch-file-organizer/data/outputs/execution_manifest_blocked_20260307_095943.